In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================


# ITDA 상품 소비기한 OCR 제출 노트북

이 노트북은 운영진의 Run All 채점 흐름을 기준으로 작성했습니다. 파인튜닝한 단일 PP-OCRv6 DET와 한국어 PP-OCRv5·숫자 중심 PP-OCRv6 REC를 사용합니다. 두 REC는 모든 후보 영역에 항상 함께 적용합니다.


In [ ]:
# 무거운 라이브러리 import 전에 CPU 스레드를 고정합니다.
N_THREADS = 4
for _name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ[_name] = str(N_THREADS)

from pathlib import Path
import sys
import cv2
import pandas as pd

BASE_DIR = Path.cwd()
sys.path.insert(0, str(BASE_DIR))
from itda_ocr.runtime import (
    ensure_weights, final_date, list_images, load_models, predict_image,
)

WEIGHTS_DIR = BASE_DIR / 'weights'
ensure_weights(WEIGHTS_DIR)
MODELS = load_models(WEIGHTS_DIR, cpu_threads=N_THREADS)
FILES = list_images(Path(INPUT_DIR)) if Path(INPUT_DIR).is_dir() else []
FIELDS = ['image_id', 'year', 'month', 'day', 'final_date']
ROWS = []
print('input:', INPUT_DIR, '| images:', len(FILES), '| det: single | rec: v5 + v6 main')


## 추론 및 중간 저장

각 검출 영역은 v5와 v6가 모두 읽습니다. 두 결과가 서로 다른 유효 날짜를 내면 confidence와 날짜 문맥을 이용해 결합하고, 실패하더라도 이미지별 예외가 전체 CSV 생성을 막지 않도록 합니다.

In [ ]:
def save_rows():
    pd.DataFrame(ROWS, columns=FIELDS).to_csv(OUTPUT_PATH, index=False)

for index, path in enumerate(FILES, 1):
    image_id = path.stem
    try:
        image = cv2.imread(str(path))
        if image is None:
            raise ValueError('image decode failed')
        ymd = predict_image(image, MODELS)
    except Exception as error:
        print('[FAIL]', image_id, type(error).__name__, error)
        ymd = ('NONE', 'NONE', 'NONE')
    year, month, day = ymd
    ROWS.append({'image_id': image_id, 'year': year, 'month': month, 'day': day, 'final_date': final_date(ymd)})
    if index == 1 or index % 25 == 0:
        save_rows()
        print(f'{index}/{len(FILES)} saved')


## 최종 저장

마지막 셀은 반드시 `OUTPUT_PATH`에 인덱스 없는 CSV를 저장합니다.

In [ ]:
df = pd.DataFrame(ROWS, columns=FIELDS)
df.to_csv(OUTPUT_PATH, index=False)
print('saved:', OUTPUT_PATH, '| shape:', df.shape)
print(df.head())